In [1]:
from openai import OpenAI, ChatCompletion
import json
import os
import getpass
import base64
from datasets import load_dataset
from io import BytesIO
from PIL import Image

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

client = OpenAI()

d:\youtube\experiments\openai-invoice-extraction\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_dataset("naver-clova-ix/cord-v2")
dataset

DatasetDict({
    train: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 800
    })
    validation: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 100
    })
    test: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 100
    })
})

In [3]:
ds_train = dataset["train"].to_pandas()
ds_val = dataset["validation"].to_pandas()
ds_test = dataset["test"].to_pandas()

In [4]:
ds_train["ground_truth"] = ds_train["ground_truth"].apply(lambda x: json.loads(x)["gt_parse"])
ds_val["ground_truth"] = ds_val["ground_truth"].apply(lambda x: json.loads(x)["gt_parse"])
ds_test["ground_truth"] = ds_test["ground_truth"].apply(lambda x: json.loads(x)["gt_parse"])

In [5]:
system_prompt = f"""You are a Vision Language Model specialized in extracting structured data from the invoice receipts.

- Your task is to analyze the provided invoice and extract the relevant information into a well-structured JSON format.
- The invoice receipt includes details such as menu, sub menu, sub total and total.
- Focus on identifying key data fields and ensuring the output adheres to the requested JSON structure.
- Fill the keys only if the information is available in the invoice.

## High-Level Problem Solving Strategy

1. Identify the main sections of the invoice: menu, sub total, and total.
2. For each section, extract the relevant data fields as specified in the schema.
3. Ensure that the output JSON is well-structured and adheres to the provided schema.
4. Do not include None or Null values in the output JSON.
5. Do not add any additional information that is not present in the invoice.
"""

In [6]:
ds_train['image'] = ds_train['image'].apply(lambda x: Image.open(BytesIO(x['bytes'])))
ds_val['image'] = ds_val['image'].apply(lambda x: Image.open(BytesIO(x['bytes'])))
ds_test['image'] = ds_test['image'].apply(lambda x: Image.open(BytesIO(x['bytes'])))

In [7]:
def encode_image(image, quality=100):
    if image.mode != 'RGB':
        image = image.convert('RGB')  # Convert to RGB
    buffered = BytesIO()
    image.save(buffered, format="JPEG", quality=quality) 
    return base64.b64encode(buffered.getvalue()).decode("utf-8") 

In [8]:
ds_train.head()

,image,ground_truth
0,<PIL.PngImagePlugin.PngImageFile image mode=RG...,"{'menu': [{'nm': 'Nasi Campur Bali', 'cnt': '1..."
1,<PIL.PngImagePlugin.PngImageFile image mode=RG...,"{'menu': [{'nm': 'SPGTHY BOLOGNASE', 'cnt': '1..."
2,<PIL.PngImagePlugin.PngImageFile image mode=RG...,"{'menu': [{'nm': 'HAKAU UDANG', 'cnt': '4', 'p..."
3,<PIL.PngImagePlugin.PngImageFile image mode=RG...,"{'menu': [{'nm': 'Bintang Bremer', 'cnt': '1',..."
4,<PIL.PngImagePlugin.PngImageFile image mode=RG...,"{'menu': {'nm': 'BASO BIHUN', 'unitprice': '43..."


In [9]:
from tqdm import tqdm

# constructing the training set
json_data = []

for idx, example in tqdm(ds_train[:100].iterrows()):
    system_message = {
        "role": "system",
        "content": [{"type": "text", "text": system_prompt}]
    }
    
    user_message = {
        "role": "user",
        "content": [
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{encode_image(example['image'], quality=50)}", "detail": "low"}}
        ]
    }
    
    assistant_message = {
        "role": "assistant",
        "content": [{"type": "text", "text": json.dumps(example["ground_truth"])}]
    }

    all_messages = [system_message] + [user_message, assistant_message]
    
    json_data.append({"messages": all_messages})

# save the JSON data to a file
with open("cord-v2-train.jsonl", "w") as f:
    for message in tqdm(json_data):
        json.dump(message, f)
        f.write("\n")

0it [00:00, ?it/s]

100it [00:04, 24.79it/s]
100%|██████████| 100/100 [00:00<00:00, 763.85it/s]


In [10]:
# sample_message = json_data[0]['messages']
# sample_message = sample_message[:2]
# print(sample_message)
# response = client.chat.completions.create(
#         model="gpt-4o-2024-08-06",
#         messages=sample_message,
#     )
# print(response.choices[0].message.content)


In [11]:
json_data = []

for idx, example in tqdm(ds_val[:20].iterrows()):
    system_message = {
        "role": "system",
        "content": [{"type": "text", "text": system_prompt}]
    }
    
    user_message = {
        "role": "user",
        "content": [
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{encode_image(example['image'], quality=50)}", "detail": "low"}}
        ]
    }
    
    assistant_message = {
        "role": "assistant",
        "content": [{"type": "text", "text": json.dumps(example["ground_truth"])}]
    }

    all_messages = [system_message] + [user_message, assistant_message]
    
    json_data.append({"messages": all_messages})



with open("cord-v2-val.jsonl", "w") as f:
    for message in json_data:
        json.dump(message, f)
        f.write("\n")


20it [00:00, 28.64it/s]


In [12]:
# sample_message = json_data[0]['messages']
# sample_message = sample_message[:2]
# print(sample_message)
# response = client.chat.completions.create(
#         model="gpt-4o-2024-08-06",
#         messages=sample_message,
#     )
# print(response.choices[0].message.content)

In [13]:
json_data = []

for idx, example in tqdm(ds_test.iterrows()):
    system_message = {
        "role": "system",
        "content": [{"type": "text", "text": system_prompt}]
    }
    
    user_message = {
        "role": "user",
        "content": [
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{encode_image(example['image'], quality=50)}", "detail": "low"}}
        ]
    }
    

    all_messages = [system_message] + [user_message]
    
    json_data.append({"messages": all_messages})

with open("cord-v2-test.jsonl", "w") as f:
    for message in json_data:
        json.dump(message, f)
        f.write("\n")


100it [00:04, 24.25it/s]


In [14]:
# sample_message = json_data[0]['messages']
# print(sample_message)
# response = client.chat.completions.create(
#         model="gpt-4o-2024-08-06",
#         messages=sample_message,
#     )
# print(response.choices[0].message.content)

In [15]:
# json.loads(response.choices[0].message.content.replace("```json", "").replace("```", "").strip())

In [16]:
# upload training file
train_file = client.files.create(
  file=open("cord-v2-train.jsonl", "rb"),
  purpose="fine-tune"
)

# upload validation file
val_file = client.files.create(
  file=open("cord-v2-val.jsonl", "rb"),
  purpose="fine-tune"
)

In [17]:
# create fine tuning job
file_train = train_file.id
file_val = val_file.id

client.fine_tuning.jobs.create(
  training_file=file_train,
  # note: validation file is optional
  validation_file=file_val,
  model="gpt-4o-2024-08-06",
  suffix="cord-v2-low"
)

FineTuningJob(id='ftjob-dIR4eRkBHVxUsJ3HoXm4x88G', created_at=1752477258, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs='auto'), model='gpt-4o-2024-08-06', object='fine_tuning.job', organization_id='org-pHH23wTxWh18imU4biwEi5Vl', result_files=[], seed=271709531, status='validating_files', trained_tokens=None, training_file='file-FLnx98KeyJXsyCxQ7geoDs', validation_file='file-BjA6GjtmxsgGpc7hnb1yxB', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs='auto'))), user_provided_suffix='cord-v2-low', usage_metrics=None, shared_with_openai=False, eval_id=None)